In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q unsloth
!pip install -q rouge-score bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3

In [2]:
import json

ADAPTER_A = "/kaggle/input/datasets/ayushsrivastava1805/thesis-adapters/Setup_A_epoch1_results/setup_a_adapters"
ADAPTER_B = "/kaggle/input/datasets/ayushsrivastava1805/thesis-adapters/Setup_B_epoch1_results/setup_b_adapters"
TEST_PATH = "/kaggle/input/datasets/ayushsrivastava1805/python-train-cot/python_test_filtered.jsonl"
MAX_SEQ_LEN = 1280

test_data = [json.loads(l) for l in open(TEST_PATH)]
print(f"Test samples: {len(test_data)}")

Test samples: 1233


In [3]:
def strip_license_header(code_str):
    lines = code_str.split("\n")
    start = 0
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith("#") or s == "" or s.startswith('"""') or s.startswith("'''"):
            start = i + 1
        else:
            break
    if start >= len(lines) - 2:
        return code_str
    return "\n".join(lines[start:])

def prompt_A(s):
    oldf = strip_license_header(s.get("oldf",""))[:1500]
    patch = s.get("patch","")[:800]
    return f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Review Comment:\n"

def prompt_B(s):
    oldf = strip_license_header(s.get("oldf",""))[:1500]
    patch = s.get("patch","")[:800]
    return f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Reasoning:\n"

# Extract final comment from Setup B output (after reasoning)
def extract_comment_B(text):
    # B output may contain "### Review Comment:" - take text after it
    if "### Review Comment:" in text:
        return text.split("### Review Comment:")[-1].strip()
    # fallback: if no marker, take the last sentence/line
    return text.strip()

In [4]:
import torch

def generate(model, tokenizer, prompt, max_new=200):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full[len(prompt):].strip()

In [5]:
from unsloth import FastLanguageModel
from tqdm import tqdm

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_A, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

preds_A = []
for s in tqdm(test_data, desc="Setup A"):
    gen = generate(model, tokenizer, prompt_A(s), max_new=150)
    preds_A.append(gen)

# Save
with open("preds_A.json", "w") as f:
    json.dump(preds_A, f)
print("Setup A done:", len(preds_A))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Starcoder2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

bigcode/starcoder2-3b does not have a padding token! Will use pad_token = <|endoftext|>.


Setup A: 100%|██████████| 1233/1233 [50:29<00:00,  2.46s/it]

Setup A done: 1233


In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_B, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

preds_B_full = []
preds_B_comment = []
for s in tqdm(test_data, desc="Setup B"):
    gen = generate(model, tokenizer, prompt_B(s), max_new=300)  # longer (reasoning)
    preds_B_full.append(gen)
    preds_B_comment.append(extract_comment_B(gen))

with open("preds_B_full.json", "w") as f:
    json.dump(preds_B_full, f)
with open("preds_B_comment.json", "w") as f:
    json.dump(preds_B_comment, f)
print("Setup B done:", len(preds_B_comment))

==((====))==  Unsloth 2026.6.9: Fast Starcoder2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

bigcode/starcoder2-3b does not have a padding token! Will use pad_token = <|endoftext|>.


Setup B: 100%|██████████| 1233/1233 [5:21:10<00:00, 15.63s/it]

Setup B done: 1233


In [7]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore
import numpy as np

actuals = [s["msg"] for s in test_data]

# ROUGE-L
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
def avg_rouge(preds):
    scores = [scorer.score(a, p)["rougeL"].fmeasure for a, p in zip(actuals, preds)]
    return np.mean(scores)

rouge_A = avg_rouge(preds_A)
rouge_B = avg_rouge(preds_B_comment)

# BERTScore
_, _, F1_A = bertscore(preds_A, actuals, lang="en", verbose=False)
_, _, F1_B = bertscore(preds_B_comment, actuals, lang="en", verbose=False)
bert_A = F1_A.mean().item()
bert_B = F1_B.mean().item()

print("="*50)
print("RESULTS (Setup A vs B)")
print("="*50)
print(f"{'Metric':<15}{'Setup A':<12}{'Setup B':<12}{'Winner'}")
print(f"{'ROUGE-L':<15}{rouge_A:<12.4f}{rouge_B:<12.4f}{'B' if rouge_B>rouge_A else 'A'}")
print(f"{'BERTScore':<15}{bert_A:<12.4f}{bert_B:<12.4f}{'B' if bert_B>bert_A else 'A'}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RESULTS (Setup A vs B)
Metric         Setup A     Setup B     Winner
ROUGE-L        0.1083      0.1093      B
BERTScore      0.8511      0.8544      B
